<br/>

<div align="center">
<span style="font-size: 2.5em;">XENON Data Pipeline Overview</span>
<br/>
<span style="font-size: 1.2em; color: gray;">Complete workflow from WimPyDD simulations to detector-level observables</span>
</div>

## Overview

This notebook provides a comprehensive tour of the data pipeline for XENON simulation-based inference:

1. **WimPyDD spectra** → Raw recoil energies from WIMP scattering
2. **CSV conversion** → PyTorch-free format for fuse compatibility
3. **Fuse output** → Detector response simulation (cS1, cS2, positions)
4. **Merged datasets** → Combined physics + detector observables
5. **Background events** → Electronic recoil (ER) backgrounds for realistic scenarios

Each section demonstrates the structure of intermediate data files.

In [ ]:
# =============================
# IMPORTS AND PATH SETTINGS
# =============================

import os
desired_root_name = "xenon-sbi"  
while os.path.basename(os.getcwd()) != desired_root_name:
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import pandas as pd
import torch

---

## 1. WimPyDD Recoil Spectra (`data/datasets/wimpy/`)

Raw Monte Carlo simulations of WIMP-nucleus scattering using WimPyDD. Each dataset contains:

- **theta**: WIMP parameters ($\log_{10} m_\chi$, $\log_{10} c_p$)
- **features**: Preprocessed observables (histogram counts, total events, top-k energies)
- **events**: Raw recoil energy arrays for each parameter point

In [ ]:
# Configuration
n = 1000
datatag = "low"
halo = "shm"

PATH = f"data/datasets/wimpy/{halo}/wimpy_n{n}_{datatag}_{halo}.pt"  
data = torch.load(PATH, weights_only=False)
print("Dataset keys:", data.keys())

dict_keys(['theta', 'features', 'events', 'logcp_range', 'logm_range', 'n_train', 'top_k', 'halo_option', 'shmpp_file', 'mc_config'])


In [ ]:
print("Feature tensor shape:", data["features"].shape)
print("Theta tensor shape:  ", data["theta"].shape)
print(f"\n→ {len(data['theta'])} parameter points, {data['features'].shape[1]} features each")

Shape of feature tensor:  torch.Size([1000, 111])
Shape of theta tensor:  torch.Size([1000, 2])


In [ ]:
# Raw recoil energy arrays (variable length)
spectra = data["events"]
print("First 4 spectra:")
for i, spec in enumerate(spectra[:4]):
    print(f"  [{i}] {len(spec)} events: {spec}")

(array([13.276286], dtype=float32),
 array([20.550322], dtype=float32),
 array([21.852188], dtype=float32),
 array([1.640459 , 1.6302305, 1.4581957, 1.1522521, 1.2741338, 1.2698387],
       dtype=float32))

In [ ]:
spectrum_lengths = [len(spec) for spec in spectra]
print(f"Average spectrum length: {np.mean(spectrum_lengths):.1f} events")
print(f"Maximum spectrum length: {max(spectrum_lengths)} events")
print(f"Zero-event spectra:      {sum(1 for s in spectra if len(s) == 0)} / {len(spectra)}")

Average spectrum length:  14.291
Maximum spectrum length:  346


**Key insight**: This analysis focuses on the **"low"** dataset (low $c_p$ regime) because:
- We're interested in low-statistics scenarios (few hundred events maximum)
- Zero-event spectra are included for proper exclusion limit calculations

---

## 2. CSV Conversion (`data/datasets/xenon/wimpy/`)

Intermediate format for fuse compatibility. Since the fuse container lacks PyTorch, recoil energies are exported to CSV:

- Each line = one spectrum (parameter point)
- Empty lines = zero-event spectra
- Format: comma-separated energy values

In [ ]:
# Configuration
n = 1000
datatag = "low"
halo = "shm"

PATH = f"data/datasets/xenon/wimpy/wimpy_n{n}_{datatag}_{halo}.csv"  

print("First 4 lines from CSV:")
with open(PATH) as f:
    for i in range(4):
        line = f.readline().strip()
        print(f"  [{i}] {line if line else '(empty)'}")

13.276286125183105

20.550321578979492

21.852188110351562

1.6404589414596558,1.6302305459976196,1.458195686340332,1.1522520780563354,1.2741338014602661,1.2698386907577515



---

## 3. Fuse Simulation Output (`data/datasets/xenon/s1s2/csv/`)

XENON detector response simulation via fuse. Outputs detector observables for each recoil event:

- **ed**: Deposited energy [keV]
- **cs1**: Corrected S1 signal (prompt scintillation)
- **cs2**: Corrected S2 signal (ionization electrons)
- **xp, yp, zp**: 3D position in the TPC [mm]

**Note**: Not all events produce measurable signals (detector thresholds, acceptance cuts). Low-energy events often have NaN values for cS1/cS2, which are filtered during training preprocessing.

In [ ]:
# Configuration (datatag dropped in filename since only "low" is simulated)
n = 1000
halo = "shm"

PATH = f"data/datasets/xenon/s1s2/csv/s1s2_n{n}_{halo}.csv"  

data = pd.read_csv(PATH)
print(f"Loaded {len(data)} events\n")
data.head()

,ed,cs1,cs2,micro_endtime,event_endtime,time_diff_ns,event_index,chunk_id,spectrum_id,local_id,xp,yp,zp
0,13.276286,8.116111,717.581970,830405084,8.321578e+08,1752766.0,0.0,0,0,0,-428.744211,-181.801917,-1174.240886
1,20.550322,40.027077,1147.202881,1074978204,1.076040e+09,1061326.0,1.0,0,1,0,146.630446,108.844709,-598.241709
2,21.852188,36.659039,1388.312744,1756768250,1.757470e+09,702150.0,2.0,0,2,0,209.868971,-361.655182,-1367.662304
3,1.640459,NaN,263.476837,1950342171,1.950964e+09,621629.0,3.0,0,3,0,-434.881182,-143.558685,-965.950656
4,1.630230,NaN,187.339600,3640738416,3.642462e+09,1723734.0,4.0,0,3,1,-66.084841,265.451383,-1317.944794


---

## 4. Merged PyTorch Datasets (`data/datasets/xenon/s1s2/pt/`)

The fuse CSV output is merged back into the original `.pt` files, adding detector observables to each parameter point.

In [ ]:
# Configuration
n = 1000
datatag = "low"
halo = "shm"

PATH = f"data/datasets/xenon/s1s2/pt/s1s2_n{n}_{halo}.pt"  
data = torch.load(PATH, weights_only=False)
print("Dataset keys:", list(data.keys()))

dict_keys(['theta', 'features', 'events', 'logcp_range', 'logm_range', 'n_train', 'top_k', 'halo_option', 'shmpp_file', 'mc_config', 'cs1cs2', 'xyz'])


In [ ]:
# Recoil energies (unchanged from original)
spectra = data["events"]
print("First 4 energy spectra:")
for i, spec in enumerate(spectra[:4]):
    print(f"  [{i}] {spec}")

(array([13.276286], dtype=float32),
 array([20.550322], dtype=float32),
 array([21.852188], dtype=float32),
 array([1.640459 , 1.6302305, 1.4581957, 1.1522521, 1.2741338, 1.2698387],
       dtype=float32))

In [ ]:
# Detector signals (cS1, cS2) for each event
cs1cs2 = data["cs1cs2"]
print("First 4 (cS1, cS2) arrays:")
for i, signals in enumerate(cs1cs2[:4]):
    print(f"  [{i}] shape {signals.shape}:\n{signals}")

[array([[  8.1161108 , 717.58197021]]),
 array([[  40.02707672, 1147.20288086]]),
 array([[  36.65903854, 1388.31274414]]),
 array([[         nan, 263.47683716],
        [         nan, 187.33959961],
        [         nan,          nan],
        [         nan,          nan],
        [         nan,          nan],
        [         nan, 175.56735229]])]

In [ ]:
# Event positions (x, y, z) in TPC
xyz = data["xyz"]
print("First 4 (x, y, z) arrays:")
for i, pos in enumerate(xyz[:4]):
    print(f"  [{i}] shape {pos.shape}:\n{pos}")

[array([[ -428.74421075,  -181.80191731, -1174.24088551]]),
 array([[ 146.63044616,  108.84470878, -598.24170914]]),
 array([[  209.86897103,  -361.65518215, -1367.66230425]]),
 array([[ -434.88118196,  -143.55868457,  -965.95065577],
        [  -66.08484076,   265.45138266, -1317.9447937 ],
        [ -452.69054838,    89.59009178,  -331.64719239],
        [ -512.15927972,    -1.65976594,  -527.18587305],
        [  406.54165537,  -206.91668595,  -815.30449233],
        [   68.83036621,   455.21009544, -1192.43036382]])]

---

## 5. Background Events (`data/datasets/xenon/s1s2/ers/`)

Electronic recoil (ER) background events from gamma interactions, simulated via fuse:

- **~10 million** independent (cS1, cS2) events
- Energy range: [0, 15] $\mathrm{keV}_{ee}$
- Used for realistic signal+background training scenarios

In [ ]:
PATH = "data/datasets/xenon/s1s2/ers/s1s2_ers.csv"
data = pd.read_csv(PATH)
print(f"Total background events: {len(data):,}\n")
data.head()

Total background events: 9,027,194



,cs1,cs2
0,69.829155,2011.4176
1,37.143440,1872.9858
2,2.926595,917.5393
3,9.634568,1373.1243
4,55.678947,2076.5540


: 